# Dari Notebook ke Produksi: Ekspor, Kuantisasi, Layanan API, dan Pemantauan

**Buku:** *Deep Learning dengan Python* — Bab 17, Deployment Model

Satu model kecil dibawa melewati seluruh jalur *deployment*: disimpan, diekspor ke SavedModel, dikonversi dan dikuantisasi ke TFLite, dibungkus API FastAPI, lalu dipantau terhadap pergeseran data.

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+. Notebook ini memakai backend **TensorFlow** (dibutuhkan oleh `tf.data`/`tf.GradientTape`/ekspor).

**Perkiraan waktu (GPU T4):** ±10 menit

**Isi notebook:**
- Format `.keras` (untuk melanjutkan pelatihan) vs `model.export()` (artefak inferensi).
- TFLite: float32, kuantisasi *dynamic range*, dan kuantisasi *full integer* dengan *representative dataset*.
- Perbandingan ukuran berkas, akurasi, dan latensi.
- Layanan FastAPI yang diuji di dalam notebook dengan `TestClient`.
- Deteksi *data drift* sederhana: uji Kolmogorov–Smirnov dan penurunan keyakinan.

In [ ]:
!pip install -q fastapi httpx

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())
import tensorflow as tf

## 1. Latih model kecil

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
from keras import layers

(x_latih, y_latih), (x_uji, y_uji) = keras.datasets.fashion_mnist.load_data()
x_latih = x_latih[..., None].astype("float32") / 255
x_uji = x_uji[..., None].astype("float32") / 255

model = keras.Sequential([keras.Input(shape=(28, 28, 1)),
                          layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
                          layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
                          layers.Flatten(), layers.Dropout(0.3),
                          layers.Dense(10, activation="softmax")])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(x_latih, y_latih, batch_size=128, epochs=5, validation_split=0.1)
print(f"Akurasi uji (Keras): {model.evaluate(x_uji, y_uji, verbose=0)[1]:.2%}")

## 2. Menyimpan vs mengekspor

In [ ]:
model.save("model.keras")                 # arsitektur + bobot + optimizer: untuk melanjutkan pelatihan
model.export("model_savedmodel")          # graf inferensi saja: untuk TF Serving, TFLite, dll.

dimuat = tf.saved_model.load("model_savedmodel")
keluaran = dimuat.serve(tf.constant(x_uji[:5]))
print("Prediksi SavedModel:", np.argmax(keluaran, axis=1), "| label:", y_uji[:5])

## 3. Konversi dan kuantisasi TFLite

In [ ]:
def konversi(mode):
    k = tf.lite.TFLiteConverter.from_saved_model("model_savedmodel")
    if mode in ("dynamic", "int8"):
        k.optimizations = [tf.lite.Optimize.DEFAULT]          # bobot -> int8
    if mode == "int8":
        def representatif():                                  # untuk kalibrasi rentang aktivasi
            for i in range(200):
                yield [x_latih[i:i + 1]]
        k.representative_dataset = representatif
        k.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    return k.convert()

varian = {mode: konversi(mode) for mode in ["float32", "dynamic", "int8"]}
for mode, isi in varian.items():
    with open(f"model_{mode}.tflite", "wb") as f:
        f.write(isi)

In [ ]:
def uji_tflite(isi, n=2000):
    interp = tf.lite.Interpreter(model_content=isi)
    interp.allocate_tensors()
    masuk, keluar = interp.get_input_details()[0], interp.get_output_details()[0]
    benar, t0 = 0, time.perf_counter()
    for i in range(n):
        interp.set_tensor(masuk["index"], x_uji[i:i + 1])
        interp.invoke()
        benar += int(interp.get_tensor(keluar["index"]).argmax() == y_uji[i])
    return benar / n, (time.perf_counter() - t0) / n * 1000

print(f"{'Varian':10s} {'Ukuran':>10s} {'Akurasi':>9s} {'Latensi':>10s}")
print(f"{'keras':10s} {os.path.getsize('model.keras') / 1024:8.0f} KB {'':>9s} {'':>10s}")
for mode, isi in varian.items():
    akurasi, latensi = uji_tflite(isi)
    print(f"{mode:10s} {len(isi) / 1024:8.0f} KB {akurasi:9.2%} {latensi:7.3f} ms")

Kuantisasi *int8* memperkecil model ±4× dengan penurunan akurasi yang biasanya sangat kecil. Selalu **ukur** akurasinya; jangan berasumsi.

## 4. Layanan API dengan FastAPI
Kode layanan ditulis ke `app.py`. Di server sungguhan, jalankan dengan `uvicorn app:app --host 0.0.0.0 --port 8000`. Di sini kita mengujinya langsung dengan `TestClient`.

In [ ]:
KODE_APP = """
import numpy as np
import tensorflow as tf
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Layanan Klasifikasi Fashion-MNIST")
interpreter = tf.lite.Interpreter(model_path="model_int8.tflite")
interpreter.allocate_tensors()
MASUK = interpreter.get_input_details()[0]
KELUAR = interpreter.get_output_details()[0]
KELAS = ["kaus", "celana", "pullover", "gaun", "mantel", "sandal", "kemeja", "sneaker", "tas", "bot"]

class Permintaan(BaseModel):
    piksel: list[list[float]]          # gambar 28x28, nilai 0-255

@app.get("/kesehatan")
def kesehatan():
    return {"status": "ok"}

@app.post("/prediksi")
def prediksi(req: Permintaan):
    x = np.asarray(req.piksel, dtype="float32").reshape(1, 28, 28, 1) / 255.0
    interpreter.set_tensor(MASUK["index"], x)
    interpreter.invoke()
    prob = interpreter.get_tensor(KELUAR["index"])[0]
    k = int(prob.argmax())
    return {"kelas": KELAS[k], "keyakinan": float(prob[k])}
"""
with open("app.py", "w") as f:
    f.write(KODE_APP)

import importlib, sys
sys.path.insert(0, os.getcwd())
import app as layanan
importlib.reload(layanan)

from fastapi.testclient import TestClient
klien = TestClient(layanan.app)
print(klien.get("/kesehatan").json())
for i in range(5):
    r = klien.post("/prediksi", json={"piksel": (x_uji[i, ..., 0] * 255).tolist()})
    print(r.status_code, r.json(), "| label asli:", layanan.KELAS[y_uji[i]])

## 5. Pemantauan: mendeteksi pergeseran data
Setelah dipasang, data masuk bisa berubah, misalnya karena kamera baru atau pencahayaan berbeda. Dua sinyal murah yang bisa dipantau tanpa label: **distribusi fitur masukan** dan **keyakinan model**.

In [ ]:
from scipy.stats import ks_2samp

def statistik(x):
    p = model.predict(x, verbose=0)
    return x.mean(axis=(1, 2, 3)), p.max(1)

acuan_kecerahan, acuan_keyakinan = statistik(x_uji[:2000])
skenario = {"normal": x_uji[2000:4000],
            "lebih terang": np.clip(x_uji[2000:4000] * 0.6 + 0.4, 0, 1),
            "terbalik (negatif)": 1 - x_uji[2000:4000]}

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for nama, x in skenario.items():
    kecerahan, keyakinan = statistik(x)
    ks = ks_2samp(acuan_kecerahan, kecerahan)
    print(f"{nama:20s} KS = {ks.statistic:.3f} (p = {ks.pvalue:.1e}) | keyakinan rata-rata {keyakinan.mean():.3f}")
    ax[0].hist(kecerahan, bins=40, alpha=0.5, label=nama)
    ax[1].hist(keyakinan, bins=40, alpha=0.5, label=nama)
ax[0].set(xlabel="Kecerahan rata-rata gambar", title="Distribusi masukan")
ax[1].set(xlabel="Keyakinan maksimum", title="Distribusi keyakinan")
for a in ax: a.legend()
plt.show()

**Catatan:** p-value KS hampir selalu "signifikan" bila sampelnya besar. Di produksi, pantau besaran statistiknya (misalnya KS > 0,1 atau PSI > 0,2) terhadap ambang yang disepakati, bukan p-value semata.